# Домашнє завдання 5: CTE та Materialized View

Тут я збираю набір ознак на рівні клієнта для ML на даних Olist. Спершу завантажую CSV і створюю typed-схему, потім рахую ознаки ланцюжком CTE, обходжу синтетичний referral-граф рекурсивним CTE, зберігаю результат у Materialized View і порівнюю плани `EXPLAIN (ANALYZE, BUFFERS)`. Дані взято з [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) на Kaggle.

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub sqlparse

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


Note: you may need to restart the kernel to use updated packages.


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
# Варіант A: якщо Kaggle credentials налаштовані
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

# Варіант B: якщо CSV вже лежать у репозиторії або course archive
# dataset_dir = '/content/goit-rdb-hw-05/data/olist'


/home/dan/hw5-verify-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/42.6M [00:00<?, ?B/s]

  2%|▏         | 1.00M/42.6M [00:00<00:30, 1.41MB/s]

  5%|▍         | 2.00M/42.6M [00:01<00:24, 1.77MB/s]

  7%|▋         | 3.00M/42.6M [00:01<00:21, 1.93MB/s]

...

 98%|█████████▊| 42.0M/42.6M [00:22<00:00, 2.27MB/s]

100%|██████████| 42.6M/42.6M [00:22<00:00, 2.31MB/s]

100%|██████████| 42.6M/42.6M [00:22<00:00, 1.97MB/s]

Extracting files...


/home/dan/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


## 1. Отримання даних і raw-шар

П'ять CSV завантажуються через KaggleHub і без змін записуються в таблиці з суфіксом `_raw`. Це лише проміжний шар: з нього дані далі переносяться в typed-таблиці PostgreSQL. Для повторного запуску потрібен або доступ до датасету через KaggleHub (варіант A), або локальна копія CSV — тоді в комірці вище треба розкоментувати шлях із варіанта B.

In [3]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)


olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [4]:
from sqlalchemy.exc import SQLAlchemyError
from IPython.display import display
import sqlparse

def execute_sql(
    sql,
    params=None,
    return_result=False,
    output_commands=True,
):
    """
    Execute a SQL script containing one or many SQL statements.

    Every statement is executed in order using the SAME database connection.

    Output:
    - SELECT / ... RETURNING -> displays a pandas DataFrame
    - INSERT / UPDATE / DELETE -> prints affected row count
    - CREATE / DROP / ALTER, etc. -> prints success
    - If output_commands=True, prints each SQL statement before execution
    - All statements are committed together if the whole script succeeds
    - On error, the transaction is rolled back

    Parameters
    ----------
    sql : str
        One SQL statement or a script containing multiple statements.

    params : dict, optional
        Named SQLAlchemy parameters, e.g. {"id": 1} for ":id".

    return_result : bool, default=False
        If True, returns a list with execution results.

    output_commands : bool, default=True
        If True, prints each SQL statement before executing it.

    Returns
    -------
    list[dict] | None
        One result dictionary per executed SQL statement if
        return_result=True, otherwise None.
    """

    statements = [
        statement.strip()
        for statement in sqlparse.split(sql)
        if statement.strip()
    ]

    outputs = []

    try:
        with engine.begin() as conn:
            for i, statement in enumerate(statements, start=1):
                print(f"\n{'=' * 70}")
                print(f"Statement {i}/{len(statements)}")
                print(f"{'=' * 70}")

                if output_commands:
                    print(statement)
                    print("-" * 70)

                result = conn.execute(
                    text(statement),
                    params or {}
                )

                if result.returns_rows:
                    rows = result.fetchall()

                    df = pd.DataFrame(
                        rows,
                        columns=result.keys()
                    )

                    display(df)

                    outputs.append({
                        "statement_number": i,
                        "statement": statement,
                        "type": "result_set",
                        "data": df,
                    })

                else:
                    rowcount = result.rowcount

                    if rowcount is not None and rowcount >= 0:
                        print(
                            "Command executed successfully. "
                            f"Rows affected: {rowcount}"
                        )
                    else:
                        print("Command executed successfully.")

                    outputs.append({
                        "statement_number": i,
                        "statement": statement,
                        "type": "command",
                        "rowcount": rowcount,
                    })

        print(
            f"\nFinished successfully: "
            f"{len(statements)} statement(s) executed."
        )

        if return_result:
            return outputs

        return None

    except SQLAlchemyError as e:
        print(f"\nSQL error while executing statement {i}:")
        print(statement)
        print(f"\n{e}")
        raise

## 2. Typed-схема й перевірка завантаження

У typed-таблицях задано типи колонок, первинні й зовнішні ключі, а також обмеження `NOT NULL` і `CHECK`. Дані з raw-шару переносяться через `INSERT ... SELECT`. Є одна дрібниця: у CSV колонки називаються `product_name_lenght` і `product_description_lenght` (з помилкою в самому датасеті), тому в `SELECT` використано саме ці назви, а в typed-схемі колонки названо правильно — `..._length`. Наприкінці рахую рядки в кожній таблиці.

In [5]:
execute_sql("""
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")


Statement 1/10
DROP TABLE IF EXISTS olist_order_items CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/10
DROP TABLE IF EXISTS olist_orders      CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 3/10
DROP TABLE IF EXISTS olist_customers   CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 4/10
DROP TABLE IF EXISTS olist_products    CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 5/10
DROP TABLE IF EXISTS olist_sellers     CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 6/10
CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INT

In [6]:
execute_sql("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
""")


Statement 1/5
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;
----------------------------------------------------------------------


Command executed successfully. Rows affected: 99441

Statement 2/5
INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;
----------------------------------------------------------------------


Command executed successfully. Rows affected: 99441

Statement 3/5
INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;
----------------------------------------------------------------------
Command executed successfully. Rows affected: 32951

Statement 4/5
INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;
----------------------------------------------------------------------
Command executed successfully. Rows affected: 3095

Statement 5/5
INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2)

Command executed successfully. Rows affected: 112650

Finished successfully: 5 statement(s) executed.


In [7]:
execute_sql("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
""")


Statement 1/1
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
----------------------------------------------------------------------


,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095



Finished successfully: 1 statement(s) executed.


Кількість рядків до і після перенесення однакова (за збереженими результатами запуску):

| Таблиця | Raw | Typed |
|---|---:|---:|
| customers | 99 441 | 99 441 |
| orders | 99 441 | 99 441 |
| order_items | 112 650 | 112 650 |
| products | 32 951 | 32 951 |
| sellers | 3 095 | 3 095 |

Під час перенесення в typed-таблиці жоден рядок не загубився, тож розбіжностей, які треба пояснювати, немає.

## 3. CTE-пайплайн ознак клієнтів

Пайплайн працює з typed-таблицями Olist. За `reference_date` беру наступний день після останнього замовлення в датасеті — так `recency_days` не залежить від того, коли саме запущено ноутбук. Кожен CTE-етап робить одну річ: окремо рахуються сума замовлення, RFM, квартал когорти, середній чек за 30 днів, частка скасувань і улюблена категорія.

In [8]:
execute_sql("""
WITH reference_date AS (
    -- Stage 0: stable reference date from the dataset

    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row and total value per order

    SELECT
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp,
        oo.order_purchase_timestamp::DATE AS order_date,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    GROUP BY
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp
),

stage2_rmf AS (
    -- Stage 2: RFM features per customer

    SELECT
        ot.customer_id,
        (SELECT as_of_date FROM reference_date)
            - MAX(ot.order_date) AS recency_days,
        COUNT(DISTINCT ot.order_id) AS frequency,
        SUM(ot.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(ot.order_purchase_timestamp)::DATE AS first_order_date
    FROM stage1_orders_total AS ot
    GROUP BY ot.customer_id
),

stage3_cohort AS (
    -- Stage 3: cohort quarter from the first order

    SELECT
        *,
        DATE_TRUNC('quarter', first_order_date)::DATE AS cohort_quarter
    FROM stage2_rmf
),

stage4_customer_30d_averages AS (
    -- Stage 4: rolling 30-day AOV for each customer's latest order
    SELECT
        customer_id,
        avg_order_value_30d
    FROM (
        SELECT
            ot.customer_id,
            AVG(ot.order_total) OVER (
                PARTITION BY ot.customer_id
                ORDER BY ot.order_purchase_timestamp
                RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
            ) AS avg_order_value_30d,
            ROW_NUMBER() OVER (
                PARTITION BY ot.customer_id
                ORDER BY ot.order_purchase_timestamp DESC, ot.order_id DESC
            ) AS rn_recent
        FROM stage1_orders_total AS ot
    ) AS ranked
    WHERE rn_recent = 1
),

stage5_cancellation_rate AS (
    -- Stage 5: canceled-order share per customer

    SELECT
        customer_id,
        COUNT(
            CASE
                WHEN order_status = 'canceled' THEN 1
            END
        )::NUMERIC(12, 2) / COUNT(*) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),

stage6_customer_revenue_by_categories AS (
    -- Stage 6: revenue by customer and product category

    SELECT
        oo.customer_id,
        op.product_category_name,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    LEFT JOIN olist_products AS op
        ON oi.product_id = op.product_id
    GROUP BY
        oo.customer_id,
        op.product_category_name
),

stage7_preferred_categories AS (
    -- Stage 7: one highest-revenue category per customer; break ties by name
    SELECT
        customer_id,
        product_category_name AS preferred_category,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC,
                         cr.product_category_name NULLS LAST
            ) AS rn
        FROM stage6_customer_revenue_by_categories AS cr
    ) AS ranked
    WHERE rn = 1
)

SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(avg.avg_order_value_30d, 0)::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,
    COALESCE(can.cancellation_rate, 0)::NUMERIC(6, 4)
        AS cancellation_rate,
    COALESCE(pref.preferred_category, '(none)')
        AS preferred_category,
    COALESCE(pref.category_revenue, 0)::NUMERIC(12, 2)
        AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON c.customer_id = coh.customer_id
LEFT JOIN stage4_customer_30d_averages AS avg
    ON c.customer_id = avg.customer_id
LEFT JOIN stage5_cancellation_rate AS can
    ON c.customer_id = can.customer_id
LEFT JOIN stage7_preferred_categories AS pref
    ON c.customer_id = pref.customer_id
ORDER BY c.customer_id
LIMIT 20;
""")


Statement 1/1
WITH reference_date AS (
    -- Stage 0: stable reference date from the dataset

    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row and total value per order

    SELECT
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp,
        oo.order_purchase_timestamp::DATE AS order_date,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    GROUP BY
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp
),

stage2_rmf AS (
    -- Stage 2: RFM features per customer

    SELECT
        ot.customer_id,
        (SELECT as_of_date FROM reference_date)
            - MAX(ot.order_date) AS recency_days,
     

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,00012a2ce6f8dcda20d059ce98491703,SP,338,1,114.74,2017-11-14,2017-10-01,114.74,0.0000,brinquedos,114.74
1,000161a058600d5901f007fab4c27140,MG,459,1,67.41,2017-07-16,2017-07-01,67.41,0.0000,beleza_saude,67.41
2,0001fd6190edaaf884bcaf3d49edf079,ES,597,1,195.42,2017-02-28,2017-01-01,195.42,0.0000,bebes,195.42
3,0002414f95344307404f0ace7a26f1d5,MG,428,1,179.35,2017-08-16,2017-07-01,179.35,0.0000,cool_stuff,179.35
4,000379cdec625522490c315e70c7a9fb,SP,199,1,107.01,2018-04-02,2018-04-01,107.01,0.0000,cama_mesa_banho,107.01
5,0004164d20a9e969af783496f3408652,SP,554,1,71.80,2017-04-12,2017-04-01,71.80,0.0000,esporte_lazer,71.80
6,000419c5494106c306a97b5635748086,RJ,230,1,49.40,2018-03-02,2018-01-01,49.40,0.0000,fashion_bolsas_e_acessorios,49.40
7,00046a560d407e99b969756e0b10f282,RJ,304,1,166.59,2017-12-18,2017-10-01,166.59,0.0000,brinquedos,166.59
8,00050bf6e01e69d5c0fd612f1bcfb69c,RS,396,1,85.23,2017-09-17,2017-07-01,85.23,0.0000,fashion_bolsas_e_acessorios,85.23
9,000598caf2ef4117407665ac33275130,MG,68,1,1255.71,2018-08-11,2018-07-01,1255.71,0.0000,pet_shop,1255.71



Finished successfully: 1 statement(s) executed.


## 4. Рекурсивний CTE: навчальний referral-граф

Реферальних зв'язків в Olist немає, тому таблицю `hw5_referrals` я заповнюю штучно. Anchor-частина через `NOT EXISTS` бере клієнтів, яких ніхто не запрошував, а рекурсивна на кожному кроці дописує запрошеного клієнта в `path` і збільшує `depth` на одиницю. Від циклів захищає перевірка, чи немає клієнта вже в `path`, а від надто довгих ланцюжків — умова `depth < 20`.

In [9]:
execute_sql("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    (
        SELECT
            customer_id
        FROM olist_customers
        ORDER BY customer_id
        OFFSET rn.row_number
        LIMIT 1
    ) AS referrer_customer_id,
    (
        SELECT customer_id
        FROM olist_customers
        ORDER BY customer_id
        OFFSET rn.row_number + 1
        LIMIT 1
    ) AS referred_customer_id
FROM (SELECT generate_series(1, 10) as row_number) AS rn;


INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    (
        SELECT
            customer_id
        FROM olist_customers
        ORDER BY customer_id
        OFFSET rn.row_number + 10
        LIMIT 1
    ) AS referrer_customer_id,
    (
        SELECT customer_id
        FROM olist_customers
        ORDER BY customer_id
        OFFSET rn.row_number * 2 + 10
        LIMIT 1
    ) AS referred_customer_id
FROM (SELECT generate_series(1, 5) as row_number) AS rn;

SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")


Statement 1/5
DROP TABLE IF EXISTS hw5_referrals CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/5
CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);
----------------------------------------------------------------------
Command executed successfully.

Statement 3/5
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    (
        SELECT
            customer_id
        FROM olist_customers
        ORDER BY customer_id
        OFFSET rn.row_number
        LIMIT 1
    ) AS referrer_customer_id,
    (
        SELECT customer_id
        FROM olist_customers
        ORDER BY customer_id
        O

,referrer_customer_id,referred_customer_id,referred_at
0,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-06 00:11:26.450966+02:00
1,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-06 00:11:26.450966+02:00
2,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-06 00:11:26.450966+02:00
3,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-06 00:11:26.450966+02:00
4,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-06 00:11:26.450966+02:00
5,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-06 00:11:26.450966+02:00
6,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-06 00:11:26.450966+02:00
7,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-06 00:11:26.450966+02:00
8,000598caf2ef4117407665ac33275130,0005aefbb696d34b3424dccd0a0e9fd0,2026-10-06 00:11:26.450966+02:00
9,0005aefbb696d34b3424dccd0a0e9fd0,00062b33cb9f6fe976afdcff967ea74d,2026-10-06 00:11:26.450966+02:00



Finished successfully: 5 statement(s) executed.


In [10]:
execute_sql("""
WITH RECURSIVE referral_depth AS (
    SELECT
        1 AS depth,
        customer_id,
        ARRAY[customer_id] AS path
    FROM olist_customers
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals
        WHERE referred_customer_id = customer_id
    )

    UNION ALL

    SELECT
        rd.depth + 1 as depth,
        r.referred_customer_id AS customer_id,
        rd.path || r.referred_customer_id AS path
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON r.referrer_customer_id = rd.customer_id
    WHERE r.referred_customer_id <> ALL(rd.path)
        AND rd.depth < 20
)
SELECT *
FROM referral_depth
WHERE depth >= 2
ORDER BY depth;
""")


Statement 1/1
WITH RECURSIVE referral_depth AS (
    SELECT
        1 AS depth,
        customer_id,
        ARRAY[customer_id] AS path
    FROM olist_customers
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals
        WHERE referred_customer_id = customer_id
    )

    UNION ALL

    SELECT
        rd.depth + 1 as depth,
        r.referred_customer_id AS customer_id,
        rd.path || r.referred_customer_id AS path
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON r.referrer_customer_id = rd.customer_id
    WHERE r.referred_customer_id <> ALL(rd.path)
        AND rd.depth < 20
)
SELECT *
FROM referral_depth
WHERE depth >= 2
ORDER BY depth;
----------------------------------------------------------------------


,depth,customer_id,path
0,2,0001fd6190edaaf884bcaf3d49edf079,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
1,2,000e943451fc2788ca6ac98a682f2f49,"[00072d033fe2e59061ae5c3aff1a2be5, 000e943451f..."
2,2,001028b78fd413e19704b3867c369d3a,"[000bf8121c3412d3057d32371c5d3395, 001028b78fd..."
3,3,0002414f95344307404f0ace7a26f1d5,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
4,4,000379cdec625522490c315e70c7a9fb,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
5,5,0004164d20a9e969af783496f3408652,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
6,6,000419c5494106c306a97b5635748086,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
7,7,00046a560d407e99b969756e0b10f282,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
8,8,00050bf6e01e69d5c0fd612f1bcfb69c,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."
9,9,000598caf2ef4117407665ac33275130,"[000161a058600d5901f007fab4c27140, 0001fd6190e..."



Finished successfully: 1 statement(s) executed.


## 5. Materialized View

У MV зберігаються вже пораховані ознаки. Унікальний індекс на `customer_id` потрібен тому, що без нього `REFRESH MATERIALIZED VIEW CONCURRENTLY` не виконається. Після створення переглядаю вміст MV і окремою коміркою запускаю concurrent refresh. Варто пам'ятати, що дані в MV актуальні лише на момент останнього оновлення.

In [11]:
execute_sql("""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH reference_date AS (
    -- Stage 0: stable reference date from the dataset

    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row and total value per order

    SELECT
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp,
        oo.order_purchase_timestamp::DATE AS order_date,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    GROUP BY
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp
),

stage2_rmf AS (
    -- Stage 2: RFM features per customer

    SELECT
        ot.customer_id,
        (SELECT as_of_date FROM reference_date)
            - MAX(ot.order_date) AS recency_days,
        COUNT(DISTINCT ot.order_id) AS frequency,
        SUM(ot.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(ot.order_purchase_timestamp)::DATE AS first_order_date
    FROM stage1_orders_total AS ot
    GROUP BY ot.customer_id
),

stage3_cohort AS (
    -- Stage 3: cohort quarter from the first order

    SELECT
        *,
        DATE_TRUNC('quarter', first_order_date)::DATE AS cohort_quarter
    FROM stage2_rmf
),

stage4_customer_30d_averages AS (
    -- Stage 4: rolling 30-day AOV for each customer's latest order
    SELECT
        customer_id,
        avg_order_value_30d
    FROM (
        SELECT
            ot.customer_id,
            AVG(ot.order_total) OVER (
                PARTITION BY ot.customer_id
                ORDER BY ot.order_purchase_timestamp
                RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
            ) AS avg_order_value_30d,
            ROW_NUMBER() OVER (
                PARTITION BY ot.customer_id
                ORDER BY ot.order_purchase_timestamp DESC, ot.order_id DESC
            ) AS rn_recent
        FROM stage1_orders_total AS ot
    ) AS ranked
    WHERE rn_recent = 1
),

stage5_cancellation_rate AS (
    -- Stage 5: canceled-order share per customer

    SELECT
        customer_id,
        COUNT(
            CASE
                WHEN order_status = 'canceled' THEN 1
            END
        )::NUMERIC(12, 2) / COUNT(*) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),

stage6_customer_revenue_by_categories AS (
    -- Stage 6: revenue by customer and product category

    SELECT
        oo.customer_id,
        op.product_category_name,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    LEFT JOIN olist_products AS op
        ON oi.product_id = op.product_id
    GROUP BY
        oo.customer_id,
        op.product_category_name
),

stage7_preferred_categories AS (
    -- Stage 7: one highest-revenue category per customer; break ties by name
    SELECT
        customer_id,
        product_category_name AS preferred_category,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC,
                         cr.product_category_name NULLS LAST
            ) AS rn
        FROM stage6_customer_revenue_by_categories AS cr
    ) AS ranked
    WHERE rn = 1
)

SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(avg.avg_order_value_30d, 0)::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,
    COALESCE(can.cancellation_rate, 0)::NUMERIC(6, 4)
        AS cancellation_rate,
    COALESCE(pref.preferred_category, '(none)')
        AS preferred_category,
    COALESCE(pref.category_revenue, 0)::NUMERIC(12, 2)
        AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON c.customer_id = coh.customer_id
LEFT JOIN stage4_customer_30d_averages AS avg
    ON c.customer_id = avg.customer_id
LEFT JOIN stage5_cancellation_rate AS can
    ON c.customer_id = can.customer_id
LEFT JOIN stage7_preferred_categories AS pref
    ON c.customer_id = pref.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features(customer_id);
""")


Statement 1/3
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
----------------------------------------------------------------------
Command executed successfully.

Statement 2/3
CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH reference_date AS (
    -- Stage 0: stable reference date from the dataset

    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row and total value per order

    SELECT
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp,
        oo.order_purchase_timestamp::DATE AS order_date,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    GROUP BY
        oo.customer_id,
        oo.order_id,
        oo.order_stat

Command executed successfully. Rows affected: 99441

Statement 3/3
CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features(customer_id);
----------------------------------------------------------------------
Command executed successfully.

Finished successfully: 3 statement(s) executed.


In [12]:
execute_sql("""
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;

SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")


Statement 1/2
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;
----------------------------------------------------------------------


,n_rows
0,99441



Statement 2/2
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
----------------------------------------------------------------------


,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66



Finished successfully: 2 statement(s) executed.


In [13]:
execute_sql("""
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
""")


Statement 1/1
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
----------------------------------------------------------------------


Command executed successfully.

Finished successfully: 1 statement(s) executed.


## 6. Плани виконання

Нижче збережено `EXPLAIN (ANALYZE, BUFFERS)` для переобчислення CTE-пайплайна і для читання з MV. Абсолютний час залежить від середовища, кешу та обсягу даних, тому при порівнянні важливіше дивитися, які операції з'являються в плані, і зважати на те, що саме лежить у MV.

In [14]:
res = execute_sql("""
EXPLAIN (ANALYZE, BUFFERS)
WITH reference_date AS (
    -- Stage 0: stable reference date from the dataset

    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row and total value per order

    SELECT
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp,
        oo.order_purchase_timestamp::DATE AS order_date,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    GROUP BY
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp
),

stage2_rmf AS (
    -- Stage 2: RFM features per customer

    SELECT
        ot.customer_id,
        (SELECT as_of_date FROM reference_date)
            - MAX(ot.order_date) AS recency_days,
        COUNT(DISTINCT ot.order_id) AS frequency,
        SUM(ot.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(ot.order_purchase_timestamp)::DATE AS first_order_date
    FROM stage1_orders_total AS ot
    GROUP BY ot.customer_id
),

stage3_cohort AS (
    -- Stage 3: cohort quarter from the first order

    SELECT
        *,
        DATE_TRUNC('quarter', first_order_date)::DATE AS cohort_quarter
    FROM stage2_rmf
),

stage4_customer_30d_averages AS (
    -- Stage 4: rolling 30-day AOV for each customer's latest order
    SELECT
        customer_id,
        avg_order_value_30d
    FROM (
        SELECT
            ot.customer_id,
            AVG(ot.order_total) OVER (
                PARTITION BY ot.customer_id
                ORDER BY ot.order_purchase_timestamp
                RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
            ) AS avg_order_value_30d,
            ROW_NUMBER() OVER (
                PARTITION BY ot.customer_id
                ORDER BY ot.order_purchase_timestamp DESC, ot.order_id DESC
            ) AS rn_recent
        FROM stage1_orders_total AS ot
    ) AS ranked
    WHERE rn_recent = 1
),

stage5_cancellation_rate AS (
    -- Stage 5: canceled-order share per customer

    SELECT
        customer_id,
        COUNT(
            CASE
                WHEN order_status = 'canceled' THEN 1
            END
        )::NUMERIC(12, 2) / COUNT(*) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),

stage6_customer_revenue_by_categories AS (
    -- Stage 6: revenue by customer and product category

    SELECT
        oo.customer_id,
        op.product_category_name,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    LEFT JOIN olist_products AS op
        ON oi.product_id = op.product_id
    GROUP BY
        oo.customer_id,
        op.product_category_name
),

stage7_preferred_categories AS (
    -- Stage 7: one highest-revenue category per customer; break ties by name
    SELECT
        customer_id,
        product_category_name AS preferred_category,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC,
                         cr.product_category_name NULLS LAST
            ) AS rn
        FROM stage6_customer_revenue_by_categories AS cr
    ) AS ranked
    WHERE rn = 1
)

SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(avg.avg_order_value_30d, 0)::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,
    COALESCE(can.cancellation_rate, 0)::NUMERIC(6, 4)
        AS cancellation_rate,
    COALESCE(pref.preferred_category, '(none)')
        AS preferred_category,
    COALESCE(pref.category_revenue, 0)::NUMERIC(12, 2)
        AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON c.customer_id = coh.customer_id
LEFT JOIN stage4_customer_30d_averages AS avg
    ON c.customer_id = avg.customer_id
LEFT JOIN stage5_cancellation_rate AS can
    ON c.customer_id = can.customer_id
LEFT JOIN stage7_preferred_categories AS pref
    ON c.customer_id = pref.customer_id
ORDER BY c.customer_id
LIMIT 20;
""", return_result=True)

print(res)

for row in res[0]['data'].itertuples(index=True, name='Row'):
    print(row)


Statement 1/1
EXPLAIN (ANALYZE, BUFFERS)
WITH reference_date AS (
    -- Stage 0: stable reference date from the dataset

    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row and total value per order

    SELECT
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp,
        oo.order_purchase_timestamp::DATE AS order_date,
        SUM(
            COALESCE(oi.price, 0) +
            COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS oo
    LEFT JOIN olist_order_items AS oi
        ON oo.order_id = oi.order_id
    GROUP BY
        oo.customer_id,
        oo.order_id,
        oo.order_status,
        oo.order_purchase_timestamp
),

stage2_rmf AS (
    -- Stage 2: RFM features per customer

    SELECT
        ot.customer_id,
        (SELECT as_of_date FROM reference_date)
            - MAX(ot.order_d

,QUERY PLAN
0,Limit (cost=76965.36..76973.31 rows=20 width=...
1,"Buffers: shared hit=11021, temp read=6713 wr..."
2,CTE stage1_orders_total
3,-> Finalize GroupAggregate (cost=16725.6...
4,Group Key: oo_1.order_id
...,...
137,...
138,Planning:
139,Buffers: shared hit=2
140,Planning Time: 0.658 ms



Finished successfully: 1 statement(s) executed.
[{'statement_number': 1, 'statement': "EXPLAIN (ANALYZE, BUFFERS)\nWITH reference_date AS (\n    -- Stage 0: stable reference date from the dataset\n\n    SELECT\n        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date\n    FROM olist_orders\n),\nstage1_orders_total AS (\n    -- Stage 1: one row and total value per order\n\n    SELECT\n        oo.customer_id,\n        oo.order_id,\n        oo.order_status,\n        oo.order_purchase_timestamp,\n        oo.order_purchase_timestamp::DATE AS order_date,\n        SUM(\n            COALESCE(oi.price, 0) +\n            COALESCE(oi.freight_value, 0)\n        )::NUMERIC(12, 2) AS order_total\n    FROM olist_orders AS oo\n    LEFT JOIN olist_order_items AS oi\n        ON oo.order_id = oi.order_id\n    GROUP BY\n        oo.customer_id,\n        oo.order_id,\n        oo.order_status,\n        oo.order_purchase_timestamp\n),\n\nstage2_rmf AS (\n    -- Stage 2: RFM features per customer\n\n    

In [15]:
execute_sql("""
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
""")


Statement 1/1
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
----------------------------------------------------------------------


,QUERY PLAN
0,Nested Loop (cost=0.83..8.98 rows=1 width=174...
1,Buffers: shared hit=5 read=3
2,-> Limit (cost=0.42..0.55 rows=1 width=32)...
3,Buffers: shared hit=4
4,-> Index Only Scan using olist_custom...
5,Heap Fetches: 1
6,Buffers: shared hit=4
7,-> Index Scan using idx_mv_olist_customer_f...
8,Index Cond: (customer_id = olist_custo...
9,Buffers: shared hit=1 read=3



Finished successfully: 1 statement(s) executed.


In [16]:
execute_sql("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")


Statement 1/2
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);
----------------------------------------------------------------------
Command executed successfully.

Statement 2/2
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
----------------------------------------------------------------------


,QUERY PLAN
0,Limit (cost=4449.01..4450.16 rows=10 width=17...
1,Buffers: shared hit=1636
2,-> Gather Merge (cost=4449.01..11175.94 ro...
3,Workers Planned: 1
4,Workers Launched: 1
5,Buffers: shared hit=1636
6,-> Sort (cost=3449.00..3595.24 rows=...
7,Sort Key: monetary DESC NULLS LAST
8,Sort Method: top-N heapsort Mem...
9,Buffers: shared hit=1636



Finished successfully: 2 statement(s) executed.


## Reflection

Плани `EXPLAIN (ANALYZE, BUFFERS)` знято після повного запуску на 99 441 клієнті Olist. CTE-запит із `LIMIT 20` виконувався 635,124 мс, а пошук одного клієнта в MV — 0,065 мс. Запити роблять різну роботу, тож це не строге порівняння, але ціну переобчислення проти точкового читання видно добре.

Найдорожче в плані обходиться блок RFM, передусім `monetary`. Для нього треба з'єднати `orders` з `order_items`, а в `stage1_orders_total` згрупувати всі 99 441 замовлення. Тут у плані є `Parallel Hash Right Join`, `Finalize GroupAggregate` і дискове сортування `external merge`. `GroupAggregate` у гілці RFM віддає перші рядки лише приблизно на 294 мс, і наступне сортування теж іде на диск. Rolling AOV за 30 днів додає `WindowAgg` з `Incremental Sort`, а для категорії потрібні ще один JOIN, агрегація й ранжування. Тобто час іде на сканування, групування й сортування, а не на різницю дат для recency.

Кешувати ці ознаки в MV має сенс, якщо їх багато разів читають для навчання чи сервінгу: важкі JOIN і агрегації виконуються один раз під час створення або refresh, а lookup іде через `Index Scan` по унікальному `customer_id`. Я б оновлював MV щодоби після завантаження нових замовлень; для статичного історичного Olist вистачить refresh на вимогу. Звичайний VIEW кращий для рідкісних запитів, коли потрібні найсвіжіші дані, а затримка не критична. Мінуси MV — зайве місце на диску й ризик прочитати застарілі значення; до того ж top-N запит до неї тут все одно дає `Parallel Seq Scan` і `Sort`.

Рекурсія зупиняється завдяки умові `referred_customer_id <> ALL(path)` і ліміту `depth < 20`. Якщо в графі є цикл A → B → A, повторний A не пройде перевірку шляху, тож обхід не стане нескінченним. Щоправда, з anchor через `NOT EXISTS` ізольований цикл без кореня в результат не потрапить. Головний компроміс цього пайплайна — швидкість повторних читань проти свіжості ознак; витрати на refresh і зберігання залежать від того, як часто треба оновлюватися.